# U6-01 · k-medias, validación y agrupamiento jerárquico

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 6, apartados 6.1 a 6.4*

Ejecuta cada celda con **Mayús + Intro**. Los conjuntos de datos vienen incluidos en scikit-learn; no hace falta descargar nada.

## 1. Los vinos y la escala (ejemplo 1)

178 vinos de tres cultivares italianos con 13 análisis químicos (Aeberhard y Forina, 1991). Agrupamos **sin usar** el cultivar; solo al final lo usamos para comprobar.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

X, y = load_wine(return_X_y=True, as_frame=True)
Z = StandardScaler().fit_transform(X)
print(X.shape, "· cultivares:", np.bincount(y))
print(X[["proline", "hue"]].agg(["min", "max"]))
for nombre, D in {"sin escalar": X.values, "estandarizado": Z}.items():
    km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(D)
    print(f"{nombre:>13}: ARI = {adjusted_rand_score(y, km.labels_):.3f}")

## 2. k-medias e inicialización (ejemplo 2)

Con una sola inicialización aleatoria el resultado depende de la semilla; k-means++ es mucho más estable.

In [ ]:
res = []
for s in range(20):
    r = KMeans(3, init="random", n_init=1, random_state=s).fit(Z).inertia_
    p = KMeans(3, init="k-means++", n_init=1, random_state=s).fit(Z).inertia_
    res.append((r, p))
res = np.array(res)
print("aleatoria : inercia entre", res[:, 0].min().round(1), "y", res[:, 0].max().round(1))
print("k-means++ : inercia entre", res[:, 1].min().round(1), "y", res[:, 1].max().round(1))

## 3. ¿Cuántos grupos? (ejemplo 3 y figura 6.2)

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

filas = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=0).fit(Z)
    l = km.labels_
    filas.append([k, km.inertia_, silhouette_score(Z, l),
                  calinski_harabasz_score(Z, l), davies_bouldin_score(Z, l)])
tabla = pd.DataFrame(filas, columns=["k", "inercia", "silueta", "CH", "DB"]).set_index("k")
print(tabla.round(3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.5))
tabla["inercia"].plot(ax=a1, marker="o", title="Método del codo")
tabla["silueta"].plot(ax=a2, marker="o", title="Silueta media")
plt.tight_layout(); plt.show()

## 4. Validación externa (ejemplo 4)

Ahora sí miramos los cultivares: tabla cruzada e índice de Rand ajustado.

In [ ]:
km = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z)
print("tamaños:", np.bincount(km.labels_), "· ARI =", round(adjusted_rand_score(y, km.labels_), 3))
pd.crosstab(y, km.labels_, rownames=["cultivar"], colnames=["grupo"])

## 5. Agrupamiento jerárquico (ejemplo 5 y figura 6.3)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
for enlace in ["ward", "complete", "average", "single"]:
    ag = AgglomerativeClustering(n_clusters=3, linkage=enlace).fit(Z)
    print(f"{enlace:>8}: ARI = {adjusted_rand_score(y, ag.labels_):6.3f} · tamaños {np.bincount(ag.labels_)}")

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
fig, ax = plt.subplots(figsize=(9, 4))
dendrogram(linkage(Z, method="ward"), no_labels=True, color_threshold=15, ax=ax)
ax.set_ylabel("altura de fusión"); plt.show()

**Ejercicio.** Repite el apartado 3 con los datos **sin escalar**. ¿Qué \(k\) señalan ahora los índices? ¿Por qué no sirven de mucho?

> Arthur, D. y Vassilvitskii, S. (2007). k-means++: The Advantages of Careful Seeding. *Proc. SODA 2007*, 1027–1035.
>
> Hubert, L. y Arabie, P. (1985). Comparing partitions. *Journal of Classification*, 2, 193–218. https://doi.org/10.1007/BF01908075